<a href="https://colab.research.google.com/github/aShamsideen/Bayesian-Optimization-for-SVM-hyperparameter-Tuning/blob/main/Music_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import ColumnTransformer

# Sample dataset (Synthetic) #
data = {
    "track_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "title":       ["Sunset Drive", "Night Pulse", "Ocean Calm", "Fire Storm",
                     "Golden Hour", "Deep Bass", "Quiet Rain", "Electric Sky"],
    "artist":      ["A", "B", "C", "D", "E", "F", "G", "H"],
    "genre":       ["pop", "electronic", "ambient", "rock",
                     "pop", "electronic", "ambient", "electronic"],
    "tempo":       [110, 128, 70, 140, 100, 130, 65, 125],
    "energy":      [0.6, 0.9, 0.2, 0.95, 0.5, 0.85, 0.15, 0.8],
    "danceability":[0.7, 0.8, 0.3, 0.6, 0.65, 0.75, 0.2, 0.78],
    "valence":     [0.7, 0.6, 0.4, 0.5, 0.75, 0.55, 0.3, 0.65],
}

df = pd.DataFrame(data)

# Build a feature matrix, Scale numeric features, One-hot encode genre #
numeric_features = ["tempo", "energy", "danceability", "valence"]
categorical_features = ["genre"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features),
    ]
)

feature_matrix = preprocessor.fit_transform(df)


# Compute similarity between all songs #
similarity_matrix = cosine_similarity(feature_matrix)


# Recommendation function #
def recommend(song_title, df, similarity_matrix, top_n=3):
    """
    Given a song title, return the top_n most similar songs.
    """
    if song_title not in df["title"].values:
        return f"'{song_title}' not found in the dataset."

    idx = df.index[df["title"] == song_title][0]
    scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity score, excluding the song itself
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    recommendations = df.iloc[[i for i, _ in scores]][["title", "artist", "genre"]].copy()
    recommendations["similarity"] = [round(score, 3) for _, score in scores]

    return recommendations.reset_index(drop=True)

if __name__ == "__main__":
    song = "Night Pulse"
    print(f"Recommendations for '{song}':\n")
    print(recommend(song, df, similarity_matrix, top_n=3))

Recommendations for 'Night Pulse':

          title artist       genre  similarity
0     Deep Bass      F  electronic       0.972
1  Electric Sky      H  electronic       0.963
2    Fire Storm      D        rock       0.508
